# **Лабораторная работа №4 Получение данных из файлов и API: унификация формата.**
---

Служба ИТ‑поддержки (opened_at, priority, duration_min)

/comments

text ← body; category ← priority; value ← duration_min

In [1]:
import json
import requests
import numpy as np
import pandas as pd

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_colwidth', None)

In [3]:
dataset = [
    {"opened_at": "2026-02-10 09:15", "priority": "high",    "duration_min": 45},
    {"opened_at": "2026-02-10 10:30", "priority": "medium",  "duration_min": 120},
    {"opened_at": "2026-02-10 09:15", "priority": "high",    "duration_min": 45},
    {"opened_at": None,               "priority": "low",     "duration_min": 60},
    {"opened_at": "2026-02-11 11:00", "priority": None,      "duration_min": 80},
    {"opened_at": "2026-02-11 12:00", "priority": "urgent",  "duration_min": 30},
    {"opened_at": "2026-02-11 12:30", "priority": "medium",  "duration_min": -20}
]
df_file = pd.DataFrame(dataset)
df_file.to_csv('local_data.csv', index=False)

In [4]:
print('saved: local_data.csv')

saved: local_data.csv


In [5]:
df_file = pd.read_csv('local_data.csv')
df_file.head()

,opened_at,priority,duration_min
0,2026-02-10 09:15,high,45
1,2026-02-10 10:30,medium,120
2,2026-02-10 09:15,high,45
3,NaN,low,60
4,2026-02-11 11:00,NaN,80


In [6]:
url_api = 'https://jsonplaceholder.typicode.com/comments'
resp = requests.get(url_api, timeout=20)
print('status_code:', resp.status_code)

status_code: 200


In [7]:
data_api = resp.json()
print('records: ', len(data_api))

records:  500


In [8]:
data_api[0]

{'postId': 1,
 'id': 1,
 'name': 'id labore ex et quam laborum',
 'email': 'Eliseo@gardner.biz',
 'body': 'laudantium enim quasi est quidem magnam voluptate ipsam eos\ntempora quo necessitatibus\ndolor quam autem quasi\nreiciendis et nam sapiente accusantium'}

In [9]:
df_api_raw = pd.DataFrame(data_api)
df_api_raw.head()

,postId,id,name,email,body
0,1,1,id labore ex et quam laborum,Eliseo@gardner.biz,laudantium enim quasi est quidem magnam voluptate ipsam eos\ntempora quo necessitatibus\ndolor quam autem quasi\nreiciendis et nam sapiente accusantium
1,1,2,quo vero reiciendis velit similique earum,Jayne_Kuhic@sydney.com,est natus enim nihil est dolore omnis voluptatem numquam\net omnis occaecati quod ullam at\nvoluptatem error expedita pariatur\nnihil sint nostrum voluptatem reiciendis et
2,1,3,odio adipisci rerum aut animi,Nikita@garfield.biz,quia molestiae reprehenderit quasi aspernatur\naut expedita occaecati aliquam eveniet laudantium\nomnis quibusdam delectus saepe quia accusamus maiores nam est\ncum et ducimus et vero voluptates excepturi deleniti ratione
3,1,4,alias odio sit,Lew@alysha.tv,non et atque\noccaecati deserunt quas accusantium unde odit nobis qui voluptatem\nquia voluptas consequuntur itaque dolor\net qui rerum deleniti ut occaecati
4,1,5,vero eaque aliquid doloribus et culpa,Hayden@althea.biz,harum non quasi et ratione\ntempore iure ex voluptates in ratione\nharum architecto fugit inventore cupiditate\nvoluptates magni quo et


### **Унификация файла CSV**

In [12]:
df_file_u = df_file.copy()
df_file_u['event_time'] = pd.to_datetime(df_file_u['opened_at'], errors='coerce')
df_file_u['category'] = df_file_u['priority']
df_file_u['value'] = df_file_u['duration_min']
df_file_u['text'] = None
df_file_u['record_id'] = ['L-' + str(i+1) for i in range(len(df_file_u))]
df_file_u['source'] = 'local'

In [13]:
df_file_u = df_file_u[['record_id', 'source', 'event_time', 'category', 'value', 'text']]
df_file_u

,record_id,source,event_time,category,value,text
0,L-1,local,2026-02-10 09:15:00,high,45,None
1,L-2,local,2026-02-10 10:30:00,medium,120,None
2,L-3,local,2026-02-10 09:15:00,high,45,None
3,L-4,local,NaT,low,60,None
4,L-5,local,2026-02-11 11:00:00,NaN,80,None
5,L-6,local,2026-02-11 12:00:00,urgent,30,None
6,L-7,local,2026-02-11 12:30:00,medium,-20,None


### **Унификация API**

In [15]:
df_api_u = df_api_raw.copy()

In [16]:
df_api_u['record_id'] = 'A-' + df_api_u['id'].astype(str)
df_api_u['source'] = 'api'
df_api_u['event_time'] = pd.Timestamp.now().floor('s')
df_api_u['category'] = df_api_u['postId'].astype(str)
df_api_u['value'] = None
df_api_u['text'] = df_api_u['body']

In [17]:
df_api_u = df_api_u[['record_id', 'source', 'event_time', 'category', 'value', 'text']]

### **Объединение**

In [18]:
unified = pd.concat([df_file_u, df_api_u], ignore_index=True)
unified.head()

,record_id,source,event_time,category,value,text
0,L-1,local,2026-02-10 09:15:00,high,45,None
1,L-2,local,2026-02-10 10:30:00,medium,120,None
2,L-3,local,2026-02-10 09:15:00,high,45,None
3,L-4,local,NaT,low,60,None
4,L-5,local,2026-02-11 11:00:00,NaN,80,None


## **Первичная валидация данных**

In [19]:
print('shape: ', unified.shape)
print('columns: ', list(unified.columns))

shape:  (507, 6)
columns:  ['record_id', 'source', 'event_time', 'category', 'value', 'text']


In [20]:
unified.dtypes

record_id             object
source                object
event_time    datetime64[ns]
category              object
value                 object
text                  object
dtype: object

In [21]:
missing_count = unified.isna().sum()
missing_percent = (unified.isna().mean() * 100).round(1)
pd.DataFrame({'missing_count': missing_count, 'missing_percent': missing_percent})

,missing_count,missing_percent
record_id,0,0.0
source,0,0.0
event_time,1,0.2
category,1,0.2
value,500,98.6
text,7,1.4


In [22]:
unified.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 507 entries, 0 to 506
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   record_id   507 non-null    object        
 1   source      507 non-null    object        
 2   event_time  506 non-null    datetime64[ns]
 3   category    506 non-null    object        
 4   value       7 non-null      object        
 5   text        500 non-null    object        
dtypes: datetime64[ns](1), object(5)
memory usage: 23.9+ KB


In [23]:
unified['value'] = pd.to_numeric(unified['value'], errors='coerce')
unified.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 507 entries, 0 to 506
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   record_id   507 non-null    object        
 1   source      507 non-null    object        
 2   event_time  506 non-null    datetime64[ns]
 3   category    506 non-null    object        
 4   value       7 non-null      float64       
 5   text        500 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(4)
memory usage: 23.9+ KB


In [24]:
dup_full = int(unified.duplicated().sum())
dup_by_id = int(unified.duplicated(subset=['record_id']).sum())
print('duplicates_full: ', dup_full)
print('duplicates_by_record_id: ', dup_by_id)

duplicates_full:  0
duplicates_by_record_id:  0


In [25]:
bad_values = unified[(unified['source'] == 'local') & (unified['value'].notna()) & (unified['value'] < 0)]
bad_values

,record_id,source,event_time,category,value,text
6,L-7,local,2026-02-11 12:30:00,medium,-20.0,None


In [27]:
validation_report = pd.DataFrame({
    'Показатель': [
        'Число строк', 'Число столбцов', 'Дубликаты (полные)', 'Дубликаты (по record_id)',
        'Пропуски (всего)', 'Некорректные value (file, value<0)'
    ],
    'Значение': [
        unified.shape[0], unified.shape[1], dup_full, dup_by_id,
        int(unified.isna().sum().sum()), len(bad_values)
    ]
})
validation_report

,Показатель,Значение
0,Число строк,507
1,Число столбцов,6
2,Дубликаты (полные),0
3,Дубликаты (по record_id),0
4,Пропуски (всего),509
5,"Некорректные value (file, value<0)",1


In [28]:
unified.to_csv('unified_data.csv', index=False)
print('saved: unified_data.csv')

saved: unified_data.csv


## **Контрольные вопросы**

**Какие преимущества и ограничения имеют файловые источники по сравнению с API‑источниками?**

Преимущества файловых источников:
* автономность - могут работать без интернета
* стабильность - данные не меняются
* скорость - отсутствие сетевых задержек
* нет лимитов - нет ограничений API (rate limits)
Ограничения файловых источников:
* устаревание - данные статичны и не обновляются
* необходимо ручное обновление
* большие данные сложно хранить локально
* отсутствие реального времени и невозможность получить актуальные данные

**Почему унификация схемы (названия/типы/время) является обязательным этапом перед анализом?**

Унификация схемы является обязательным этапов перед анализом, поскольку:
* разные колонки нельзя склеить
* невозможно посчитать суммы, если поля названы по-разному
* строка '100' != число 100, т.е. ошибки в расчетах
* разные форматы дат приводят к невозможности сортировки и фильтрации

**Какие элементы включает первичная валидация данных и что она позволяет выявить?**

Элементы первичной валидации данных:

(1) *структура*
   
* наличие обязательных колонок
* количество строк/столбцов
* дубликаты колонок
  
(2) *типы данных*
* соответствие ожидаемым типам (int, float, date, str)
* корректность преобразований
  
(3) *пропуски (NaN)*
* количество пропусков по колонкам
* паттерны пропусков
  
(4) *диапазоны значений*
* минимум/максимум (выбросы)
* отрицательные значения там, где их не должно быть
  
(5) *уникальность*
* проверка уникальных ID
* дубликаты строк

Это позволяет выявить:
* битые данные (поврежденный файл)
* неполные данные (загрузка не всех полей)
* дубликаты
* аномалии
* несоответствие формату

**Какие ошибки типизации могут привести к некорректным агрегированиям и визуализациям?**

Основные ошибки типизации, приводящие к некорректным результатам:
* проблема 1: числа как строки
* проблема 2: даты как строки
* проблема 3: категории как числа

**Какие дополнительные проверки качества данных целесообразны в прикладных проектах?**

Дополнительные проверки качества:

(1) *бизнес правила*

* цена должна быть > 0
* дата заказа <= дата доставки
* Email содержит @
  
(2) *статистические аномалии*

* выбросы (за 3 σ от среднего)
* резкие изменения (временные ряды)

(3) *целостность связей*

* все user_id существуют в таблице users
* нет "потерянных" транзакций

(4) *временная согласованность*

* данные за ожидаемый период
* нет дат из будущего

(5) *полнота покрытия*

* все обязательные категории присутствуют
* данные за все дни периода (нет пропущенных дней)
